# Urdu POS Tagger — BiLSTM + CRF

Deep-learning sequence labeller for Urdu Part-of-Speech tagging.

| Component | Detail |
|-----------|--------|
| **Dataset** | UD Urdu Treebank — 4,323 train / 516 val / 535 test sentences, 17 UPOS tags |
| **Model** | Word Embeddings → 2-layer BiLSTM → Linear → CRF decoder |
| **Framework** | PyTorch + `pytorch-crf` |
| **Target accuracy** | ≥ 90 % |

---

In [ ]:
# Run once; restart kernel afterwards if packages were newly installed
!pip install -q torch pytorch-crf datasets scikit-learn matplotlib seaborn tqdm pandas

## 1. Imports & Configuration

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF
from datasets import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
from tqdm.notebook import tqdm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Hyper-parameters ──────────────────────────────────────────────────────
EMBEDDING_DIM = 128
HIDDEN_DIM    = 256
NUM_LAYERS    = 2
DROPOUT       = 0.3
BATCH_SIZE    = 32
EPOCHS        = 25
LR            = 1e-3
CLIP          = 1.0
PAD_WORD      = '<PAD>'
UNK_WORD      = '<UNK>'

## 2. Load UD Urdu Treebank

The [UD Urdu Treebank](https://universaldependencies.org/treebanks/ur_udtb/index.html) is loaded directly from HuggingFace Hub. It contains ~5,400 annotated sentences with 17 universal POS tags.

In [ ]:
print('Downloading UD Urdu Treebank …')
raw = load_dataset('universal-dependencies/universal_dependencies', 'ur_udtb')
print(raw)

# Extract UPOS tag names from the dataset's ClassLabel feature
upos_feature = raw['train'].features['upos'].feature
TAG_NAMES = upos_feature.names          # e.g. ['_', 'ADJ', 'ADP', ...]
NUM_TAGS  = len(TAG_NAMES)
tag2idx   = {t: i for i, t in enumerate(TAG_NAMES)}
idx2tag   = {i: t for t, i in tag2idx.items()}

print(f'\nUPOS tags ({NUM_TAGS}): {TAG_NAMES}')
print(f'Train  : {len(raw["train"]):,} sentences')
print(f'Validation: {len(raw["validation"]):,} sentences')
print(f'Test   : {len(raw["test"]):,} sentences')

## 3. Data Exploration

In [ ]:
# ── Token & sentence statistics ──────────────────────────────────────────
splits = {'train': raw['train'], 'validation': raw['validation'], 'test': raw['test']}
for split, ds in splits.items():
    tok_count = sum(len(s['tokens']) for s in ds)
    avg_len   = tok_count / len(ds)
    print(f'{split:12s}: {len(ds):5,} sents | {tok_count:7,} tokens | avg len {avg_len:.1f}')

# Show three example sentences
print('\n── Sample sentences ─────────────────────────────────────────────────')
for ex in raw['train'].select(range(3)):
    pairs = list(zip(ex['tokens'], [idx2tag[u] for u in ex['upos']]))
    print('  '.join(f'{w}/{t}' for w, t in pairs))
    print()

In [ ]:
# ── Tag frequency distribution ───────────────────────────────────────────
tag_counter = Counter()
for ex in raw['train']:
    for u in ex['upos']:
        tag_counter[idx2tag[u]] += 1

labels, counts = zip(*sorted(tag_counter.items(), key=lambda x: -x[1]))

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(labels, counts, color=sns.color_palette('Set2', len(labels)))
ax.bar_label(bars, fmt='%d', fontsize=8)
ax.set_title('POS Tag Frequency Distribution — UD Urdu Train Set', fontsize=14)
ax.set_xlabel('UPOS Tag')
ax.set_ylabel('Token Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Top-5 tags:', ', '.join(f'{l}={c}' for l, c in zip(labels[:5], counts[:5])))

In [ ]:
# ── Sentence length distribution ────────────────────────────────────────
lengths = [len(ex['tokens']) for ex in raw['train']]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lengths, bins=40, color='steelblue', edgecolor='white')
ax.axvline(np.mean(lengths), color='red', linestyle='--', label=f'Mean = {np.mean(lengths):.1f}')
ax.axvline(np.percentile(lengths, 95), color='orange', linestyle='--',
           label=f'95th pct = {np.percentile(lengths, 95):.0f}')
ax.set_title('Sentence Length Distribution (train)', fontsize=14)
ax.set_xlabel('Tokens per sentence')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Preprocessing — Vocabulary & Dataset

In [ ]:
# ── Build word vocabulary from training data ─────────────────────────────
word_counter = Counter()
for ex in raw['train']:
    word_counter.update(ex['tokens'])

# PAD=0, UNK=1, then all words sorted by frequency
vocab = [PAD_WORD, UNK_WORD] + [w for w, _ in word_counter.most_common()]
word2idx = {w: i for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)
PAD_IDX    = word2idx[PAD_WORD]
UNK_IDX    = word2idx[UNK_WORD]

print(f'Vocabulary size: {VOCAB_SIZE:,}')
print(f'PAD index: {PAD_IDX}   UNK index: {UNK_IDX}')

In [ ]:
# ── PyTorch Dataset ───────────────────────────────────────────────────────
class UrduPOSDataset(Dataset):
    def __init__(self, hf_split):
        self.data = [
            (
                [word2idx.get(w, UNK_IDX) for w in ex['tokens']],
                ex['upos'],
            )
            for ex in hf_split
        ]

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]


def collate_fn(batch):
    """Pad sequences in a batch to the same length."""
    word_seqs, tag_seqs = zip(*batch)
    max_len = max(len(s) for s in word_seqs)
    words = torch.zeros(len(batch), max_len, dtype=torch.long)
    tags  = torch.zeros(len(batch), max_len, dtype=torch.long)
    mask  = torch.zeros(len(batch), max_len, dtype=torch.bool)
    for i, (ws, ts) in enumerate(zip(word_seqs, tag_seqs)):
        n = len(ws)
        words[i, :n] = torch.tensor(ws, dtype=torch.long)
        tags[i, :n]  = torch.tensor(ts, dtype=torch.long)
        mask[i, :n]  = True
    return words, tags, mask


train_loader = DataLoader(UrduPOSDataset(raw['train']),      batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(UrduPOSDataset(raw['validation']), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(UrduPOSDataset(raw['test']),       batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

## 5. Model Architecture — BiLSTM-CRF

```
Input tokens
      │
  Embedding (128-d)
      │
  Dropout (0.3)
      │
  BiLSTM × 2 layers  (256-d hidden, 128-d per direction)
      │
  Linear (256 → num_tags)
      │
  CRF decoder  ← learns valid tag-transition constraints
      │
  Predicted tag sequence
```

The CRF layer replaces a plain softmax: it models dependencies between consecutive tags (e.g., a VERB is more likely after a NOUN than after PUNCT) and is the standard decoder for state-of-the-art sequence labellers.

In [ ]:
class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, num_tags, embedding_dim,
                 hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.dropout    = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim // 2,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc  = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def _emit(self, x):
        emb = self.dropout(self.embedding(x))       # (B, T, E)
        out, _ = self.lstm(emb)                     # (B, T, H)
        return self.fc(out)                          # (B, T, num_tags)

    def forward(self, x, tags, mask):
        """Return negative log-likelihood loss (scalar)."""
        emissions = self._emit(x)
        return -self.crf(emissions, tags, mask=mask, reduction='mean')

    def predict(self, x, mask):
        """Return list-of-lists of predicted tag indices."""
        with torch.no_grad():
            emissions = self._emit(x)
        return self.crf.decode(emissions, mask=mask)


model = BiLSTM_CRF(
    vocab_size    = VOCAB_SIZE,
    num_tags      = NUM_TAGS,
    embedding_dim = EMBEDDING_DIM,
    hidden_dim    = HIDDEN_DIM,
    num_layers    = NUM_LAYERS,
    dropout       = DROPOUT,
    pad_idx       = PAD_IDX,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTrainable parameters: {total_params:,}')

## 6. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True
)


def run_epoch(loader, train=False):
    if train:
        model.train()
    else:
        model.eval()

    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for words, tags, mask in loader:
            words, tags, mask = words.to(device), tags.to(device), mask.to(device)
            if train:
                optimizer.zero_grad()
            loss = model(words, tags, mask)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                optimizer.step()
            total_loss += loss.item()

            # Accuracy
            preds = model.predict(words, mask)
            for pred_seq, true_seq, m in zip(preds, tags, mask):
                L = m.sum().item()
                correct += sum(p == t for p, t in
                               zip(pred_seq[:L], true_seq[:L].tolist()))
                total   += L

    return total_loss / len(loader), correct / total

In [ ]:
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_acc = 0.0
best_state   = None

print(f'{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>9} | {'Val Acc':>8}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)

    train_losses.append(tr_loss);  val_losses.append(vl_loss)
    train_accs.append(tr_acc);     val_accs.append(vl_acc)

    scheduler.step(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}
        marker = '  ← best'
    else:
        marker = ''

    print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>8.2%} | {vl_loss:>9.4f} | {vl_acc:>7.2%}{marker}')

# Restore best weights
model.load_state_dict(best_state)
print(f'\nBest validation accuracy: {best_val_acc:.2%}')

In [ ]:
# ── Learning curves ──────────────────────────────────────────────────────
epochs_x = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_x, train_losses, label='Train', marker='o', markersize=3)
ax1.plot(epochs_x, val_losses,   label='Val',   marker='s', markersize=3)
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Neg. Log-Likelihood')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_x, [a * 100 for a in train_accs], label='Train', marker='o', markersize=3)
ax2.plot(epochs_x, [a * 100 for a in val_accs],   label='Val',   marker='s', markersize=3)
ax2.set_title('Accuracy per Epoch')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('BiLSTM-CRF Training Curves', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Evaluation on Held-Out Test Set

In [ ]:
# ── Collect all predictions and true labels ──────────────────────────────
model.eval()
all_true, all_pred = [], []

with torch.no_grad():
    for words, tags, mask in test_loader:
        words, tags, mask = words.to(device), tags.to(device), mask.to(device)
        preds = model.predict(words, mask)
        for pred_seq, true_seq, m in zip(preds, tags, mask):
            L = m.sum().item()
            all_pred.extend(pred_seq[:L])
            all_true.extend(true_seq[:L].tolist())

all_true_names = [idx2tag[i] for i in all_true]
all_pred_names = [idx2tag[i] for i in all_pred]

test_acc = sum(p == t for p, t in zip(all_pred, all_true)) / len(all_true)
print(f'Test Accuracy : {test_acc:.4%}')
print(f'Total tokens  : {len(all_true):,}')

In [ ]:
# ── Per-tag classification report ───────────────────────────────────────
present_tags = sorted(set(all_true_names) | set(all_pred_names))
report = classification_report(
    all_true_names, all_pred_names,
    labels=present_tags, zero_division=0
)
print(report)

In [ ]:
# ── Confusion matrix (normalised) ────────────────────────────────────────
cm = confusion_matrix(all_true_names, all_pred_names, labels=present_tags)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    pd.DataFrame(cm_norm, index=present_tags, columns=present_tags),
    annot=True, fmt='.2f', cmap='Blues',
    linewidths=0.4, ax=ax, vmin=0, vmax=1
)
ax.set_title(f'Normalised Confusion Matrix (Test) — Accuracy {test_acc:.2%}', fontsize=14)
ax.set_ylabel('True Tag')
ax.set_xlabel('Predicted Tag')
plt.tight_layout()
plt.show()

## 8. Error Analysis

In [ ]:
# ── Most-confused tag pairs ──────────────────────────────────────────────
errors = Counter()
for t, p in zip(all_true_names, all_pred_names):
    if t != p:
        errors[(t, p)] += 1

print(f'Total errors : {sum(errors.values()):,} / {len(all_true):,} tokens')
print(f'\nTop-15 confused pairs (true → predicted):')
print(f'{"True":>8} {"Predicted":>10} {"Count":>7}')
print('-' * 30)
for (true_t, pred_t), cnt in errors.most_common(15):
    print(f'{true_t:>8} → {pred_t:<10} {cnt:>6}')

In [ ]:
# ── Bar chart of per-tag error rates ─────────────────────────────────────
true_total  = Counter(all_true_names)
tag_errors  = Counter(t for t, p in zip(all_true_names, all_pred_names) if t != p)
error_rates = {
    tag: tag_errors.get(tag, 0) / true_total[tag]
    for tag in sorted(true_total, key=lambda x: -true_total[x])
    if true_total[tag] >= 20
}

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(error_rates.keys(), [v * 100 for v in error_rates.values()],
              color=sns.color_palette('Reds_r', len(error_rates)))
ax.bar_label(bars, fmt='%.1f%%', fontsize=8)
ax.set_title('Per-Tag Error Rate (test set)', fontsize=13)
ax.set_xlabel('UPOS Tag')
ax.set_ylabel('Error Rate (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Live Demo — Tag New Urdu Sentences

In [ ]:
def tag_sentence(sentence: str):
    """Tag an Urdu sentence (space-separated tokens)."""
    tokens = sentence.split()
    indices = torch.tensor(
        [[word2idx.get(w, UNK_IDX) for w in tokens]],
        dtype=torch.long, device=device
    )
    mask = torch.ones(1, len(tokens), dtype=torch.bool, device=device)
    pred_ids = model.predict(indices, mask)[0]
    return list(zip(tokens, [idx2tag[i] for i in pred_ids]))


demo_sentences = [
    'میں نے کتاب پڑھی ۔',
    'وہ لاہور میں رہتا ہے ۔',
    'احمد نے کھانا کھایا ۔',
    'بچے سکول جاتے ہیں ۔',
    'پاکستان ایک خوبصورت ملک ہے ۔',
]

for sent in demo_sentences:
    print(f'\nInput : {sent}')
    tagged = tag_sentence(sent)
    print('Output: ' + '  '.join(f'{w}[{t}]' for w, t in tagged))

In [ ]:
# ── Colour-coded table output ────────────────────────────────────────────
TAG_COLORS = {
    'NOUN':'#AED6F1','VERB':'#A9DFBF','ADJ':'#F9E79F','ADV':'#FAD7A0',
    'PRON':'#D7BDE2','PROPN':'#85C1E9','ADP':'#F1948A','AUX':'#82E0AA',
    'DET':'#F8C471','NUM':'#76D7C4','PUNCT':'#CCD1D1','CCONJ':'#F0B27A',
    'SCONJ':'#E59866','PART':'#A3E4D7','INTJ':'#EAF2FF','SYM':'#D5DBDB','X':'#E8DAEF',
}

sent = demo_sentences[0]
tagged = tag_sentence(sent)
rows = [{'Token': w, 'Predicted Tag': t} for w, t in tagged]
df = pd.DataFrame(rows)

def color_row(row):
    c = TAG_COLORS.get(row['Predicted Tag'], '#FFFFFF')
    return [f'background-color: {c}'] * len(row)

print('Colour-coded tagging for:', sent)
df.style.apply(color_row, axis=1)